# M4 — Concurrency Systems + Translations (UFC/BJJ edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paiml/big-o-python-to-rust/blob/main/notebooks/m4-systems.ipynb)

Systems-level complexity: cache misses for linear roster scans, work-depth speedup (Brent's theorem) for parallel fight predictions, external-sort I/O for a UFC fight-history database. Plus the course concurrency translations: generator -> iterator (4.1.1), subprocess -> Command (4.2.1), threading -> rayon (4.3.1).

## Cache complexity — `(M, B)` model

Linear scan of `n` fighters with line size `B` (fighters per cache line) incurs
`ceil(n / B)` cache misses. Smaller working sets (a single weight class) stay
hot in cache; full roster scans saturate L2/L3.

In [1]:
from dataclasses import dataclass


@dataclass
class CacheModel:
    lines: int
    line_size: int


def cache_misses(n: int, m: CacheModel) -> int:
    return -(-n // m.line_size)  # ceiling division


model = CacheModel(lines=64, line_size=8)
assert cache_misses(8, model) == 1
assert cache_misses(64, model) == 8
assert cache_misses(65, model) == 9  # one extra line for the lone overflow
print(f"cache        : roster=64, line_size=8 -> {cache_misses(64, model)} misses")

cache        : roster=64, line_size=8 -> 8 misses


## Lesson 4.3.1 — `threading` -> `rayon` (Brent's theorem)

Brent's theorem: with `work` total work and `depth` critical path, `p` processors
give `min(p, work / depth)` speedup. Python `threading` is GIL-bound; Rust
`rayon` parallelizes across cores at zero ceremony. The work-depth model proves
the upper bound.

In [2]:
def parallel_speedup(work: int, depth: int, p: int) -> float:
    return float(min(p, work // depth))


# Predicting fights: 1024 simulations, each depth-8 (8 rounds)
assert parallel_speedup(1024, 8, 1) == 1.0
assert parallel_speedup(1024, 8, 8) == 8.0
assert parallel_speedup(1024, 8, 256) == 128.0  # depth-limited
# Monotonicity: more cores cannot slow you down
speeds = [parallel_speedup(1024, 8, p) for p in (1, 2, 4, 8, 16, 64)]
assert all(speeds[i] <= speeds[i + 1] for i in range(len(speeds) - 1))
print(f"brent (rayon): speedups (p=1..64) = {speeds}")

brent (rayon): speedups (p=1..64) = [1.0, 2.0, 4.0, 8.0, 16.0, 64.0]


## External-sort I/O — `subprocess` -> `Command` (lesson 4.2.1)

Sorting a UFC fight-history database too large to fit in memory: external merge
sort with memory `M` and block size `B` runs in `O((n/B) * log_{M/B} (n/B))` I/O
operations. Spawning the sort via `Command` (vs Python `subprocess`) gets you
proper exit codes + no string-injection footgun.

In [3]:
from math import ceil, log


def external_sort_io_cost(n: int, m: int, b: int) -> int:
    blocks = ceil(n / b)
    if blocks <= 1 or m // b <= 1:
        return blocks
    return 2 * blocks * ceil(log(blocks, m // b))


# Monotonicity: more memory => fewer (or equal) I/O ops
cost_small_m = external_sort_io_cost(1_000_000, 4096, 64)
cost_big_m = external_sort_io_cost(1_000_000, 1_000_000, 64)
assert cost_big_m <= cost_small_m
print(f"external     : I/O cost @ M=4096   = {cost_small_m}")
print(f"external     : I/O cost @ M=10^6   = {cost_big_m} (fewer passes)")

external     : I/O cost @ M=4096   = 93750
external     : I/O cost @ M=10^6   = 31250 (fewer passes)


## Lesson 4.1.1 — `generator` -> `Iterator` (streaming fight history)

Python generators and Rust iterators are siblings: lazy, single-pass, allocation-
free until you collect. A streaming scan over a 10M-fight history file holds O(1)
memory in either language — but the Rust version compiles to a tight loop with no
heap allocations at all.

In [4]:
def stream_wins(fight_history: list[dict]):
    """Generator: yield winners lazily without materializing a list."""
    for fight in fight_history:
        yield fight["winner"]


history = [
    {"winner": "Khabib", "loser": "McGregor"},
    {"winner": "Adesanya", "loser": "Pereira"},
    {"winner": "Jones", "loser": "Cormier"},
]

# Streaming: O(1) memory, never materializes a list
first_win = next(stream_wins(history))
assert first_win == "Khabib"

# Even when we count, the iterator stays lazy (sum doesn't allocate)
total_wins = sum(1 for _ in stream_wins(history))
assert total_wins == 3
print(f"streaming    : {total_wins} fights scanned with O(1) memory")

streaming    : 3 fights scanned with O(1) memory


---
**Rust port:** [`m4-systems/src/lib.rs`](../m4-systems/src/lib.rs) hosts the same functions with monotonicity proptest invariants. Course lessons 4.1.1, 4.2.1, 4.3.1.